# Compute analysis/forecast windowed TCC, RMSE, and bias maps

Reads model output and observations and caches per-member plus ensemble-mean map diagnostics for each lead window.


In [ ]:
from pathlib import Path
INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_out")

groups = ("Jan2012",)
run_by_group = {"Dec2011": ("da",), "Jan2012": ("fc",), "Jun2012": ("fc",)}
selected = {"ERA5": ("TREFHT",), "GPM": ("PRECT",)}
force_compute = False
verbose = True


window_scheme = "biweekly"  # "biweekly", "weekly", "15day", or "month"


In [ ]:
from pathlib import Path
import sys

repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))

from configs.output_paths import workflow_output_dirs
from configs.s2s_config import EXPERIMENT_GROUPS, OBS_VARIABLES, PCC_REGIONS, TCC_REGIONS, LEAD_WINDOWS, WEEKLY_WINDOWS, FIFTEEN_DAY_WINDOWS, MONTH_WINDOWS, WINDOW_SCHEMES, AVAILABLE_PCC_METRICS, AVAILABLE_PCC_MAP_METRICS, AVAILABLE_TCC_METRICS
from util.s2s_experiments import build_experiments
from util.s2s_observations import OBS_REGISTRY, get_obs_file


## Notes: available selections

- Experiment groups: Dec2011 (analysis), Jan2012, and Jun2012 (forecasts).
- Runs: da for Dec2011; fc (standard forecast) or wc (water-cycle forecast) for forecast groups.
- Observations and variables: `ERA5` supports UBOT, VBOT, PRECT, PSL, T925, Q925, U850, V850, T850, Q850, Z500, OMEGA850/500/200, U/V500, U/V200, FLDS/C, FLNS/C, FSDS/C, FSNS/C, LHFLX, SHFLX, QFLX, TREFHT, TS, PRECSL, PRECC, and PRECL; `GPCP` and `GPM` support PRECT; `NOAA-OLR` supports FLUT.
- Regions: GLOBAL, land/ocean where applicable, NINO3.4, MAR_CONT, CONUS, NH_POLAR.
- Existing NetCDF diagnostics are reused. Set `force_compute = True` to rebuild selected products.

Available map metrics: TCC, RMSE, and bias. Lead windows default to weeks 1–2, 3–4, 5–6, and 7–8.

Exact base-map fields are `TCC`, `RMSE`, and `BIAS` for members and `TCC_ensmean`, `RMSE_ensmean`, and `BIAS_ensmean` for the ensemble mean. For each base metric, regional outputs provide `<METRIC>_regional_member`, `<METRIC>_member_ens_mean`, `<METRIC>_member_ens_std`, `<METRIC>_member_qpostXX`, `<METRIC>_member_qpreXX`, `<METRIC>_regional_ensmean`, valid-weight and valid-fraction fields, plus `region_area_weight_sum`. `XX` is each configured quantile percentile.


In [ ]:
from util.s2s_time_windows import build_windows_from_start
from util.s2s_tcc_metrics import FileCollectionConfig, S2SFileCollector, WindowedTCCBaseConfig, S2SWindowedTCCBaseBuilder

data_dir, figure_dir = workflow_output_dirs("fcst_s2s_tcc", f"{window_scheme}_metrics", output_root=DIAGNOSTIC_OUTPUT_ROOT)
exp_dict = build_experiments(str(INPUT_DATA_ROOT))
lead_windows = WINDOW_SCHEMES[window_scheme]
for group in groups:
    meta = EXPERIMENT_GROUPS[group]
    windows = build_windows_from_start(meta["init_date"], lead_windows)
    group_dir = data_dir / group; group_dir.mkdir(parents=True, exist_ok=True)
    builder = S2SWindowedTCCBaseBuilder(WindowedTCCBaseConfig(windows=windows, regions=[dict(name="GLOBAL",lat=(-90,90),lon=(0,360))], out_dir=str(group_dir), overwrite=force_compute, verbose=verbose))
    for run in run_by_group.get(group, (meta["run"],)):
        for obs, variables in selected.items():
            var_map = {v: OBS_VARIABLES[obs][v] for v in variables}
            cfg = FileCollectionConfig(group=group, freq=meta["freq"], run=run, obs=obs, period=meta["period"], model_template="%(var)s.%(ens)s.%(year)d.nc")
            collector = S2SFileCollector(exp_list=EXPERIMENT_GROUPS, exp_dict=exp_dict, obs_registry=OBS_REGISTRY, s2s_var_dict=var_map, get_obs_file_func=get_obs_file)
            files = collector.collect(cfg, vars_to_process=list(variables), verbose=verbose)
            for var in variables:
                if var not in files: continue
                for exp, exp_files in files[var].items():
                    for ens in exp_files["model"]:
                        builder.build_member_file(files, group=group, freq=meta["freq"], run=run, obs_name=obs, period=meta["period"], var=var, exp=exp, ens=ens, obs_var_override=var_map)
                    builder.build_ensmean_file(files, group=group, freq=meta["freq"], run=run, obs_name=obs, period=meta["period"], var=var, exp=exp, obs_var_override=var_map)

